In [ ]:
# ── 1. Install extra packages not included in Kaggle by default ───────────
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip_install('deap')      # Genetic Algorithm library
pip_install('tabulate')  # Pretty tables in compare_models
print('✅  Extra packages installed')

In [ ]:
# ── 2. Copy source files to /kaggle/working so they are importable ────────
import os, shutil, glob

SRC_DATASET = '/kaggle/input/ransomware-src'
WORK_DIR    = '/kaggle/working'

if not os.path.isdir(SRC_DATASET):
    raise FileNotFoundError(
        f'Source dataset not found at {SRC_DATASET}.\n'
        'Please add the "ransomware-src" dataset to this notebook.'
    )

for py_file in glob.glob(f'{SRC_DATASET}/*.py'):
    dst = os.path.join(WORK_DIR, os.path.basename(py_file))
    shutil.copy2(py_file, dst)
    print(f'  copied: {os.path.basename(py_file)}')

# Make working dir importable
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print('\n✅  Source files ready')

In [ ]:
# ── 3. Verify dataset path ────────────────────────────────────────────────
RISS_PATH = '/kaggle/input/riss-dataset/RISS.csv'

if not os.path.exists(RISS_PATH):
    # Try to auto-find it
    found = glob.glob('/kaggle/input/**/RISS.csv', recursive=True)
    if found:
        RISS_PATH = found[0]
        print(f'Found RISS.csv at: {RISS_PATH}')
    else:
        raise FileNotFoundError(
            'RISS.csv not found. Add the "riss-dataset" dataset to this notebook.'
        )

import pandas as pd
df_check = pd.read_csv(RISS_PATH, header=None, nrows=3)
print(f'✅  RISS.csv  →  shape check: {df_check.shape[1]} columns (first 3 rows)')

In [ ]:
# ── 4. Suppress TF noise ──────────────────────────────────────────────────
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
print(f'TensorFlow {tf.__version__}')

In [ ]:
# ── 5. Import project modules ─────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

from load_data import load_data

print('✅  Project modules imported')


In [ ]:
# ── 6. Define MODEL class (mirrors main.py exactly, paths adjusted for Kaggle) ──
BASE_OUT = '/kaggle/working'

class MODEL:
    def __init__(self, data_path):
        self.saved_model_path = os.path.join(BASE_OUT, 'models', 'saved', 'PremierLeague.keras')
        self.saved_data_path  = os.path.join(BASE_OUT, 'data',   'processed', 'PremierLeague_processed_data.csv')
        self.data_path  = data_path
        self.X_train = self.X_val = self.X_test = None
        self.y_train = self.y_val = self.y_test = None
        self.n_features = self.n_classes = None
        self.model   = None
        self.scaler  = StandardScaler()
        self.label_encoder  = LabelEncoder()
        self.result_encoder = LabelEncoder()
        self.home_encoder   = LabelEncoder()
        self.away_encoder   = LabelEncoder()
        # GA parameters
        self.population_size = 6    # increase from 2 for a real Kaggle run
        self.generations     = 10   # increase from 1 for a real Kaggle run
        self.crossover_prob  = 0.85
        self.mutation_prob   = 0.15
        # Results
        self.best_individual = None
        self.best_fitness    = 0
        self.best_metrics    = {}
        self.logbook         = None  # filled by GA.py after eaSimple

obj = MODEL(RISS_PATH)
print('✅  MODEL object created')
print(f'   GA settings: population={obj.population_size}, generations={obj.generations}')


In [ ]:
# ── 7. Load & preprocess data ─────────────────────────────────────────────
# idx='4' → RISS branch (no header, last col = label, feature selection applied)
load_data(obj, idx='4')

print(f'\nSplit summary:')
print(f'  X_train : {obj.X_train.shape}  |  classes in y_train : {np.unique(obj.y_train)}')
print(f'  X_val   : {obj.X_val.shape}')
print(f'  X_test  : {obj.X_test.shape}')
print(f'  n_features after selection : {obj.n_features}')
print(f'  n_classes                  : {obj.n_classes}')

In [ ]:
# ── 8. Initialise GA run — save original arrays & persistent result store ──
import copy, os, pickle
from GA             import run_ga_optimization
from eval           import evaluate_best_model
from print_resault  import display_results
from compare_models import rank_models, print_final_ranking, generate_charts, save_results_log

CHART_DIR = BASE_OUT
PARTIAL_RESULTS_PATH = os.path.join(BASE_OUT, 'ga_partial_results.pkl')

# Save the flat (2-D) arrays once so each model cell can restore them
_X_train_2d = obj.X_train.copy()
_X_val_2d   = obj.X_val.copy()
_X_test_2d  = obj.X_test.copy()

def _save_partial_results():
    with open(PARTIAL_RESULTS_PATH, 'wb') as f:
        pickle.dump(all_results, f)
    print(f'💾  Partial results saved → {PARTIAL_RESULTS_PATH}')

def _load_partial_results():
    if os.path.exists(PARTIAL_RESULTS_PATH):
        with open(PARTIAL_RESULTS_PATH, 'rb') as f:
            return pickle.load(f)
    return {}

# Shared dict — each model cell below adds its entry here
all_results = _load_partial_results()
if all_results:
    print(f'📦  Loaded {len(all_results)} previously finished model(s)')
else:
    print('📦  No previous partial results found')

def _run_one_model(obj, model_type):
    """Run GA + evaluate for a single model, store result in all_results."""
    print(f'\n{"═"*70}')
    print(f'  🚀  GA — {model_type}')
    print('═'*70)

    # Restore flat arrays (RNN/LSTM reshape them; this prevents cross-contamination)
    obj.X_train = _X_train_2d.copy()
    obj.X_val   = _X_val_2d.copy()
    obj.X_test  = _X_test_2d.copy()

    # Reset best-* fields
    obj.best_individual = None
    obj.best_fitness    = 0.0
    obj.best_metrics    = {}
    obj.logbook         = None

    try:
        exec_time = run_ga_optimization(obj, test=model_type)
        evaluate_best_model(obj, test=model_type)
        display_results(obj, exec_time, test=model_type)

        all_results[model_type] = {
            'best_individual': copy.deepcopy(obj.best_individual),
            'best_fitness'   : obj.best_fitness,
            'best_metrics'   : copy.deepcopy(obj.best_metrics),
            'execution_time' : exec_time,
            'logbook'        : copy.deepcopy(getattr(obj, 'logbook', None)),
            'error'          : None,
        }
        _save_partial_results()
        print(f'✅  {model_type} done  |  F1={obj.best_metrics.get("f1_score", 0):.4f}')

    except Exception as exc:
        print(f'❌  {model_type} failed: {exc}')
        all_results[model_type] = {
            'best_individual': None,
            'best_fitness'   : 0.0,
            'best_metrics'   : {'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0},
            'execution_time' : 0.0,
            'logbook'        : None,
            'error'          : str(exc),
        }
        _save_partial_results()

print('✅  Init done — run each model cell below individually')
print('   Finished models are saved after every run, so progress is not lost.')


In [ ]:
# ── 9. MLP — GA optimisation ──────────────────────────────────────────────
_run_one_model(obj, 'MLP')


In [ ]:
# ── 10. CNN — GA optimisation ─────────────────────────────────────────────
_run_one_model(obj, 'CNN')


In [ ]:
# ── 11. RNN — GA optimisation ─────────────────────────────────────────────
_run_one_model(obj, 'RNN')


In [ ]:
# ── 12. DNN — GA optimisation ─────────────────────────────────────────────
_run_one_model(obj, 'DNN')


In [ ]:
# ── 13. LSTM — GA optimisation ────────────────────────────────────────────
_run_one_model(obj, 'LSTM')


In [ ]:
# ── 14. Build ranking from completed runs ─────────────────────────────────
all_results = _load_partial_results()

if not all_results:
    raise ValueError('No model results found yet. Run at least one model cell first.')

ranking = rank_models(all_results)
print('✅  Ranking built')
print(f'   Models included: {list(all_results.keys())}')
print(f'   Total completed : {len(all_results)}')


In [ ]:
# ── 15. Print final comparison report ─────────────────────────────────────
if 'ranking' not in globals():
    all_results = _load_partial_results()
    ranking = rank_models(all_results)

print_final_ranking(ranking)


In [ ]:
# ── 16. Generate charts at the end ────────────────────────────────────────
if 'ranking' not in globals():
    all_results = _load_partial_results()
    ranking = rank_models(all_results)

generate_charts(all_results, ranking, CHART_DIR)
print(f'✅  Charts generated in {CHART_DIR}')


In [ ]:
# ── 17. Save the comparison log file ──────────────────────────────────────
if 'ranking' not in globals():
    all_results = _load_partial_results()
    ranking = rank_models(all_results)

log_path = save_results_log(all_results, ranking, CHART_DIR)
print(f'✅  Log saved → {log_path}')


In [ ]:
# ── 18. Display saved charts inline ───────────────────────────────────────
import glob
from IPython.display import display, Image

chart_paths = sorted(glob.glob(os.path.join(CHART_DIR, 'chart_*.png')))
print(f'Found {len(chart_paths)} chart(s)')
for path in chart_paths:
    print(f'  {os.path.basename(path)}')
    display(Image(filename=path))
